# N18 · Speculative Decoding Acceptance

**配套 lab**：L08.7 · Speculative Decoding

**目标**：把"投机解码什么时候赢、什么时候亏"具体化。理解：

1. acceptance rate 与 expected speedup 的公式关系
2. draft cost ratio 的影响
3. K（num_speculative_tokens）选择
4. in-domain vs OOD prompt 的 acceptance 差异
5. ITL bimodal 分布与 p99 退化

**No-GPU 友好**。

## 1. 运行前预测

1. acceptance_rate = 0.5, draft_cost_ratio = 0.25 时 expected speedup ≈ ____
2. acceptance_rate < draft_cost_ratio 时 speedup > 1 还是 < 1？
3. n-gram 模式与 draft model 模式哪个 draft_gpu_mem_gb 更高？
4. OOD prompt 的 acceptance 通常比 in-domain 低 ____ 倍
5. K=4 vs K=8 哪个对 acceptance 衰减更敏感？

## 2. Acceptance vs speedup 公式

In [ ]:
from mini_infra.vllm.spec_decode.acceptance_tracker import AcceptanceTracker, acceptance_summary
from mini_infra.vllm.spec_decode.draft_runner import spec_decode_summary
from mini_infra.vllm.spec_decode.ngram import ngram_summary, ngram_candidates

# 公式：speedup ≈ 1 / ((1 - rate) + draft_cost_ratio)
print("acceptance_rate vs expected speedup (draft_cost_ratio=0.25):")
for rate in (0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.85, 0.95):
    speedup = 1.0 / max((1 - rate) + 0.25, 1e-9)
    marker = " <-- 临界（speedup=1）" if 0.99 < speedup < 1.01 else ""
    marker = " <-- 接近上限" if rate >= 0.85 else marker
    print(f"  rate={rate:.2f}  speedup={speedup:.3f}{marker}")

**结论**：当 draft_cost_ratio=0.25 时，acceptance < 0.25 几乎没有收益（speedup ≈ 1）。
draft_cost 越高，临界 acceptance 越高。所以 ngram（draft cost ≈ 0）即使 acceptance=0.2 也有 speedup ≈ 1.25。

## 3. AcceptanceTracker 滑动窗口

In [ ]:
tracker = AcceptanceTracker(window=8)
for accept_count in [3, 2, 4, 1, 3, 0, 2, 3]:
    tracker.add(accept_count, proposed=4)
print(f"accepted history: [3, 2, 4, 1, 3, 0, 2, 3] (proposed=4 each)")
print(f"acceptance_rate = {tracker.rate()}")
print(f"expected_speedup (draft_cost=0.25) = {tracker.expected_speedup(0.25)}")
print(f"expected_speedup (draft_cost=0.10) = {tracker.expected_speedup(0.10)}  <- ngram")
print(f"expected_speedup (draft_cost=0.50) = {tracker.expected_speedup(0.50)}  <- 大 draft model")

## 4. ngram 候选生成

In [ ]:
long_prompt = "the model should cite sources and the model should explain assumptions".split()
for suffix in [["the", "model", "should"], ["sources", "and"], ["never", "appears"]]:
    candidates = ngram_candidates(long_prompt, suffix, max_tokens=4)
    print(f"  suffix={suffix} -> candidates={candidates}")

print()
print("ngram_summary 默认输入:")
import json
print(json.dumps(ngram_summary(), indent=2, ensure_ascii=False))

**观察**：
- 长 prompt + 重复模板 → ngram 命中率高
- 短 prompt 或无重复结构 → 命中率为 0
- ngram 完全不占 GPU（draft_gpu_mem_gb=0）

## 5. In-domain vs OOD acceptance

In [ ]:
for mode in ("ngram", "draft"):
    for domain in ("in_domain", "out_of_domain"):
        s = spec_decode_summary(mode=mode, concurrency=1, domain=domain)
        print(f"mode={mode:6s}  domain={domain:14s}  "
              f"accept_rate={s['acceptance_rate']:.3f}  "
              f"speedup={s['speedup']:.3f}  "
              f"itl_p50={s['itl_ms_p50']:.2f}ms")

## 6. 高并发下 p99 退化

In [ ]:
for concurrency in (1, 16, 32, 64, 128):
    s = spec_decode_summary(mode="draft", concurrency=concurrency, domain="in_domain")
    p99_p50_ratio = s['itl_ms_p99'] / s['itl_ms_p50']
    print(f"concurrency={concurrency:3d}  itl_p50={s['itl_ms_p50']:6.2f}  itl_p99={s['itl_ms_p99']:6.2f}  "
          f"p99/p50={p99_p50_ratio:.2f}  speedup={s['speedup']:.2f}")

**观察**：concurrency ≥ 64 时 p99/p50 比值显著上升（公式中 1.8x 系数）。
这就是"mean ITL 改善 + p99 退化"的根因。生产 SLO 通常按 p99，所以 spec 在高并发下需谨慎。

## 7. 运行后反思

| 预测项 | 你的预测 | 实际 | ✓/✗ |
|---|---|---|---|
| rate=0.5, cost=0.25 speedup | | ~1.33 | |
| rate < cost speedup | | < 1（亏） | |
| ngram vs draft mem | | draft 高 | |
| OOD vs in-domain accept | | OOD 显著低 | |
| K=4 vs K=8 衰减 | | K=8 更敏感 | |

**回到 lab**：
- 监控 acceptance_rate 必须含滑动窗口（生产用 1000+，本课用 32）
- 高 acceptance + 低并发开 spec；OOD 或高并发关 spec
- speedup 公式与实测偏差 > 30% 去 ticket `spec_low_acceptance_001`
- p99 退化去 ticket `spec_unstable_p99_003`
- draft OOM 去 ticket `spec_draft_oom_002`